# Surgical tool detection: train on Kaggle

Before running:
1. **Settings → Accelerator → GPU T4 x2** (or P100).
2. **Settings → Internet → On** (needs a phone-verified Kaggle account). Used to clone the repo and install packages.
3. **Add Input** → add each dataset you want to train on. They appear under `/kaggle/input/<name>/`.
   Start with `jocelyndumlao/surgical-tools` (the Mendeley 9-class set).

For a full training run use **Save Version → Save & Run All (Commit)**, so it keeps running after you close the tab.
Everything written under `/kaggle/working/` is kept as the notebook's output.

In [ ]:
%cd /kaggle/working
!rm -rf surgical-tool-detection
!git clone -q https://github.com/Ahmed-Hammadi/surgical-tool-detection
%cd surgical-tool-detection
!pip install -q -r requirements.txt
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## 1. Look at the input datasets
Check the folder names and where the label files are before preparing.

In [ ]:
!ls /kaggle/input
!find /kaggle/input -maxdepth 3 | head -60
!find /kaggle/input -type f | sed 's/.*\.//' | sort | uniq -c | sort -rn | head

## 2. Prepare
List one folder per dataset in `RAW`. If the script stops with "Labels not in configs/classes.yaml",
add those labels to `configs/classes.yaml` in the GitHub repo (alias, `ignore` or `exclude_images_with`), then re-run from the top.

In [ ]:
RAW = "/kaggle/input/surgical-tools"   # several datasets: "/kaggle/input/a /kaggle/input/b"
!python src/prepare_dataset.py --raw {RAW} --out data/yolo
!cat data/yolo/coverage_report.md

## 3. Train
YOLO11n at 640 px. `batch=-1` picks the largest batch that fits the GPU.

In [ ]:
!python src/train.py --set batch=-1 workers=4 plots=true

If a session was cut off, add the previous notebook version's output as an input and resume from its `last.pt`:

`!yolo train resume model=/kaggle/input/<your-notebook-output>/surgical-tool-detection/runs/yolo11n_640/weights/last.pt`

## 4. Evaluate

In [ ]:
!python src/evaluate.py --weights runs/yolo11n_640/weights/best.pt

## 5. Export for the phone (optional)
Not yet tried on Kaggle. If the install changes the PyTorch version and breaks the session, skip this cell and export later from the saved `best.pt`.

In [ ]:
!pip install -q -r requirements-export.txt
!python src/export.py --weights runs/yolo11n_640/weights/best.pt --targets litert-int8 litert-w8a32 onnx

## 6. Pack the results
Download `results.zip` from the Output panel (weights, reports, phone files). The prepared dataset is removed so the output stays small.

In [ ]:
!zip -q -r /kaggle/working/results.zip runs/yolo11n_640/weights runs/yolo11n_640/results.csv runs/eval runs/export mobile data/yolo/coverage_report.md data/yolo/split_report.json data/yolo/borderline_pairs.csv 2>/dev/null
!rm -rf data/yolo
!ls -la /kaggle/working/results.zip